# Interpretação e fairness

Bancada de trabalho: a lógica reutilizável fica em `src/`; aqui corre-se, explora-se e
guarda-se o que o `Projeto_Final.ipynb` vai ler (`reports/experiments.csv`, `reports/figures/`).
Regras da modelação: [docs/modelacao_base.md](../docs/modelacao_base.md).

Importância por permutação (`sklearn.inspection.permutation_importance`, PR-AUC) e métricas por idade, raça e sexo, tudo sobre as previsões OOF do treino. Sem explicação por doente (limitação a declarar no model card).

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # os caminhos de src/ (data/, reports/, models/) são relativos à raiz

import pandas as pd

from src.models.baselines import candidate_models
from src.data.population import follow_up_eligible
from src.models.evaluate import metrics_by_group, oof_permutation_importance
from src.pipeline import load_model_data

data = load_model_data()
best = candidate_models()["hist_gradient_boosting"]
oof = pd.read_csv(ROOT / "reports" / "oof_v1.csv")["oof_score"].to_numpy()  # gravado no 04
elig = follow_up_eligible(data.X_train).to_numpy()  # lista de follow-up só com altas elegíveis (D21)

## Importância por permutação

Descida da PR-AUC na validação de cada fold quando a feature é baralhada (5 repetições); média e desvio entre os 5 folds.

In [2]:
imp = oof_permutation_importance(best, data)
imp.to_csv(ROOT / "reports" / "permutation_importance_v1.csv")
imp.head(15).round(4)

,importance,std
number_inpatient,0.0519,0.0078
discharge_disposition_id_desc,0.0496,0.0039
payer_code,0.0077,0.0011
medical_specialty,0.0056,0.0018
n_visits_prior,0.0052,0.0018
diag_1_group,0.0048,0.0015
admission_source_id_desc,0.0046,0.0016
diabetes_complicated,0.0026,0.0012
number_emergency,0.0025,0.0012
age_band,0.0024,0.0016


## Fairness por grupo

Top 10% calculado em todas as altas **elegíveis** (D21, OOF); por grupo vê-se a taxa de sinalizados e o recall entre as elegíveis. Grupos com menos de 100 encontros não se reportam.

In [3]:
{g: metrics_by_group(data.y_train[elig], oof[elig], data.X_train.loc[elig, g]).round(3)
 for g in ["age_band", "race", "gender"]}

{'age_band':            n  positive_rate  flagged_rate  recall  precision  pr_auc
 group                                                               
 0        127          0.024         0.000   0.000        NaN   0.062
 1        539          0.059         0.083   0.312      0.222   0.138
 2       1259          0.147         0.261   0.665      0.374   0.421
 3       2714          0.110         0.178   0.441      0.273   0.274
 4       6417          0.098         0.121   0.337      0.272   0.245
 5      10529          0.082         0.087   0.239      0.224   0.187
 6      12529          0.095         0.090   0.202      0.213   0.172
 7      12063          0.097         0.086   0.191      0.216   0.175
 8       5628          0.099         0.086   0.179      0.207   0.162
 9        631          0.125         0.062   0.101      0.205   0.165,
 'race':                      n  positive_rate  flagged_rate  recall  precision  pr_auc
 group                                                     